# 8주차 A회차 실습 과제: 뉴스 기사 토픽 모델링

이 노트북은 `docs/ch8A.md`의 코딩 시연을 그대로 옮긴 것이 아니라, 하나의 과제를 처음부터 끝까지 수행하도록 구성한 실습용 노트북이다.

## 과제

가상의 한국어 뉴스 기사 묶음에서 BERTopic으로 숨겨진 주제를 찾고, 각 주제의 핵심 단어와 대표 문서를 근거로 주제 이름을 붙인다.

## 산출물

- `data/output/ch8A_topic_info.csv`: 발견된 주제 요약
- `data/output/ch8A_document_topics.csv`: 문서별 주제 배정 결과
- `data/output/ch8A_topic_keywords.json`: 주제별 핵심 단어
- `data/output/ch8A_topic_barchart.html`: 주제별 핵심 단어 시각화
- `data/output/ch8A_topic_heatmap.html`: 주제 간 유사도 시각화
- `data/output/ch8A_documents.html`: 문서 임베딩 지도


## 0. 환경 확인

루트에서 `setup_env.py`를 실행하면 `.venv`가 생성되고 기본 패키지가 설치된다. Jupyter에서는 `2026NLP` 커널을 선택한 뒤 이 노트북을 실행한다.

```bash
cd /Users/callii/Documents/2026-1/2026NLP
python setup_env.py
source .venv/bin/activate
python -m ipykernel install --user --name 2026nlp --display-name "2026NLP"
```


In [1]:
# ============================================================
# 루트 가상환경 및 chapter8 패키지 확인
# ============================================================
from pathlib import Path
import importlib.util
import subprocess
import sys

ROOT_DIR = Path("/Users/callii/Documents/2026-1/2026NLP")
ROOT_VENV = ROOT_DIR / ".venv"
REQUIREMENTS = ROOT_DIR / "practice" / "chapter8" / "requirements.txt"

if Path(sys.prefix).resolve() != ROOT_VENV.resolve():
    raise RuntimeError(
        "현재 노트북 커널이 루트 .venv가 아닙니다. "
        "루트에서 `python setup_env.py`를 실행한 뒤 Jupyter 커널을 `2026NLP`로 선택하세요.\n"
        f"현재 Python: {sys.executable}\n"
        f"필요한 가상환경: {ROOT_VENV}"
    )

required_modules = {
    "bertopic": "bertopic",
    "sentence_transformers": "sentence-transformers",
    "umap": "umap-learn",
    "hdbscan": "hdbscan",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "plotly": "plotly",
}

missing = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    print(f"설치되지 않은 chapter8 패키지: {', '.join(missing)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS)])
else:
    print("chapter8 패키지 확인 완료")

print(f"사용 중인 Python: {sys.executable}")
print(f"가상환경: {sys.prefix}")


chapter8 패키지 확인 완료
사용 중인 Python: /Users/callii/Documents/2026-1/2026NLP/.venv/bin/python
가상환경: /Users/callii/Documents/2026-1/2026NLP/.venv


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    import torch
    from bertopic import BERTopic
    from hdbscan import HDBSCAN
    from sentence_transformers import SentenceTransformer
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
    from umap import UMAP
except ImportError as exc:
    raise ImportError(
        "필수 패키지가 설치되어 있지 않습니다. "
        "루트에서 `python -m pip install -r practice/chapter8/requirements.txt`를 실행하세요."
    ) from exc

SEED = 42
np.random.seed(SEED)
BASE_DIR = Path.cwd()
if BASE_DIR.name != "chapter8":
    BASE_DIR = Path("/Users/callii/Documents/2026-1/2026NLP/practice/chapter8")

OUTPUT_DIR = BASE_DIR / "data" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"실행 디바이스: {device}")
print(f"산출물 저장 위치: {OUTPUT_DIR}")


## 1. 과제 데이터 로드

`data/input/ch8A_news_topics.csv`에 미리 준비된 한국어 뉴스 기사 데이터를 불러온다. `true_topic`은 모델 학습에는 사용하지 않고, 마지막 검증 단계에서만 비교한다.


In [ ]:
INPUT_PATH = BASE_DIR / "data" / "input" / "ch8A_news_topics.csv"

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"과제 데이터 파일을 찾을 수 없습니다: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH)
required_columns = {"text", "true_topic"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"데이터에 필요한 컬럼이 없습니다: {sorted(missing_columns)}")

df = df.dropna(subset=["text"]).reset_index(drop=True)
docs = df["text"].astype(str).tolist()

print(f"데이터 파일: {INPUT_PATH}")
print(f"문서 수: {len(docs)}")
print("주제별 문서 수:")
print(df["true_topic"].value_counts().sort_index())
display(df.head(8))


## 2. BERTopic 모델 구성과 학습

파이프라인은 `문서 임베딩 → UMAP 차원 축소 → HDBSCAN 클러스터링 → c-TF-IDF 주제 표현` 순서로 실행된다.


In [ ]:
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2", device=device)

umap_model = UMAP(
    n_neighbors=10,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=SEED,
)

hdbscan_model = HDBSCAN(
    min_cluster_size=8,
    min_samples=3,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    token_pattern=r"(?u)[가-힣A-Za-z0-9]{2,}",
    min_df=2,
    ngram_range=(1, 2),
)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="multilingual",
    calculate_probabilities=True,
    verbose=False,
)

topics, probabilities = topic_model.fit_transform(docs)
df["topic"] = topics

topic_info = topic_model.get_topic_info()
display(topic_info)

n_topics = len([t for t in set(topics) if t != -1])
n_noise = sum(1 for t in topics if t == -1)
print(f"발견된 주제 수: {n_topics}")
print(f"노이즈 문서 수: {n_noise}")


## 3. 주제별 핵심 단어와 대표 문서 확인

아래 결과를 보고 각 주제가 무엇을 의미하는지 직접 이름을 붙인다.


In [ ]:
representative_docs_by_topic = topic_model.get_representative_docs()
topic_keywords = {}
topic_keyword_scores = {}
valid_topic_ids = sorted(t for t in df["topic"].unique() if t != -1)
grouped_texts = [" ".join(df.loc[df["topic"] == topic_id, "text"]) for topic_id in valid_topic_ids]

keyword_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)[가-힣A-Za-z0-9]{2,}",
    min_df=1,
    ngram_range=(1, 2),
    max_features=500,
)
topic_term_matrix = keyword_vectorizer.fit_transform(grouped_texts)
terms = keyword_vectorizer.get_feature_names_out()

for row_idx, topic_id in enumerate(valid_topic_ids):
    scores = topic_term_matrix[row_idx].toarray().ravel()
    top_indices = scores.argsort()[::-1][:8]

    keyword_pairs = [(terms[i], float(scores[i])) for i in top_indices if scores[i] > 0]
    keywords = [word for word, score in keyword_pairs]
    topic_keywords[int(topic_id)] = keywords
    topic_keyword_scores[int(topic_id)] = keyword_pairs

    representative_docs = representative_docs_by_topic.get(topic_id, [])[:2]
    print("=" * 80)
    print(f"주제 {topic_id}")
    print(f"핵심 단어: {', '.join(keywords)}")
    print("대표 문서:")
    for doc in representative_docs:
        print(f"- {doc}")

with open(OUTPUT_DIR / "ch8A_topic_keywords.json", "w", encoding="utf-8") as f:
    json.dump(topic_keywords, f, ensure_ascii=False, indent=2)


## 4. 실제 주제와 모델 주제 비교

실제 수업에서는 정답 레이블이 없는 경우가 많다. 여기서는 실습 검증을 위해 생성 시 사용한 `true_topic`과 BERTopic의 `topic`을 비교한다.


In [ ]:
comparison = pd.crosstab(df["topic"], df["true_topic"], margins=True)
display(comparison)

topic_label_guess = {}
for topic_id in sorted(t for t in df["topic"].unique() if t != -1):
    topic_docs = df[df["topic"] == topic_id]
    majority_label = topic_docs["true_topic"].value_counts().idxmax()
    purity = topic_docs["true_topic"].value_counts(normalize=True).max()
    topic_label_guess[int(topic_id)] = majority_label
    print(f"주제 {topic_id}: 추정 이름 = {majority_label}, 순도 = {purity:.2f}, 문서 수 = {len(topic_docs)}")


## 5. 특정 문서의 주제 분석

문서 하나를 골라 모델이 어떤 주제로 분류했는지 확인하고, 분류가 타당한지 해석한다.


In [ ]:
doc_idx = 0
doc_topic = int(df.loc[doc_idx, "topic"])
doc_true_topic = df.loc[doc_idx, "true_topic"]
doc_text = df.loc[doc_idx, "text"]

print(f"문서 번호: {doc_idx}")
print(f"문서 내용: {doc_text}")
print(f"실제 주제: {doc_true_topic}")
print(f"모델 주제: {doc_topic}")
print(f"모델 주제 이름 추정: {topic_label_guess.get(doc_topic, '노이즈')}")

if probabilities is not None:
    probs = np.asarray(probabilities)
    if probs.ndim == 2 and doc_idx < len(probs):
        top_prob_idx = probs[doc_idx].argsort()[::-1][:5]
        print("\n상위 주제 확률:")
        for rank, topic_pos in enumerate(top_prob_idx, start=1):
            print(f"{rank}. topic_position={topic_pos}, probability={probs[doc_idx][topic_pos]:.3f}")
    elif probs.ndim == 1 and doc_idx < len(probs):
        print(f"\n배정 신뢰도: {probs[doc_idx]:.3f}")


## 6. 시각화 저장

Plotly 기반 HTML 파일은 브라우저에서 열어 상호작용하며 확인할 수 있다.


In [ ]:
visualization_outputs = []

try:
    import plotly.express as px

    keyword_rows = []
    for topic_id, pairs in topic_keyword_scores.items():
        label = topic_label_guess.get(topic_id, f"주제 {topic_id}")
        for word, score in pairs:
            keyword_rows.append({"topic": f"주제 {topic_id}: {label}", "word": word, "score": score})

    keyword_df = pd.DataFrame(keyword_rows)
    fig = px.bar(
        keyword_df,
        x="score",
        y="word",
        color="topic",
        facet_col="topic",
        facet_col_wrap=2,
        orientation="h",
        height=900,
        title="주제별 핵심 단어 TF-IDF 점수",
    )
    fig.update_yaxes(matches=None, autorange="reversed")
    fig.update_xaxes(matches=None)
    fig.update_layout(showlegend=False)
    output_path = OUTPUT_DIR / "ch8A_topic_barchart.html"
    fig.write_html(output_path)
    visualization_outputs.append(output_path)
except Exception as exc:
    print(f"barchart 저장 실패: {exc}")

try:
    fig = topic_model.visualize_heatmap(top_n_topics=min(8, n_topics))
    output_path = OUTPUT_DIR / "ch8A_topic_heatmap.html"
    fig.write_html(output_path)
    visualization_outputs.append(output_path)
except Exception as exc:
    print(f"heatmap 저장 실패: {exc}")

try:
    embeddings = topic_model._extract_embeddings(docs)
    fig = topic_model.visualize_documents(docs, embeddings=embeddings)
    output_path = OUTPUT_DIR / "ch8A_documents.html"
    fig.write_html(output_path)
    visualization_outputs.append(output_path)
except Exception as exc:
    print(f"documents 저장 실패: {exc}")

for path in visualization_outputs:
    print(f"저장 완료: {path}")


## 7. 결과 저장과 제출용 질문

아래 파일을 저장한 뒤, 다음 질문에 답한다.

1. BERTopic이 발견한 주제 수는 몇 개인가?
2. 각 주제의 핵심 단어와 대표 문서를 근거로 주제 이름을 어떻게 붙일 수 있는가?
3. 노이즈 문서는 몇 개이며, 노이즈가 생긴 이유는 무엇이라고 해석할 수 있는가?
4. 실제 주제와 모델 주제가 잘 맞지 않는 사례가 있다면 왜 그런가?


In [ ]:
topic_info_path = OUTPUT_DIR / "ch8A_topic_info.csv"
document_topics_path = OUTPUT_DIR / "ch8A_document_topics.csv"

topic_info.to_csv(topic_info_path, index=False, encoding="utf-8-sig")
df.to_csv(document_topics_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {topic_info_path}")
print(f"저장 완료: {document_topics_path}")
print(f"저장 완료: {OUTPUT_DIR / 'ch8A_topic_keywords.json'}")

display(df.head(10))
